# Inferring Column Purpose and Meaning

**Notebook Version**: 1.0.0  
**Compatible with DBMCP**: 0.1.0+  
**Last Updated**: 2026-01-26  
**Test Database Version**: 1.0  
**Estimated Time**: 15 minutes  
**Difficulty**: Intermediate

## Overview

This notebook demonstrates how to analyze database columns to infer their purpose. Many legacy databases have cryptic column names like `FLG_1`, `STATUS_CD`, or `AMT_3`. The Column Analyzer helps you understand what these columns represent by examining:

- Data patterns and value distributions
- Null percentages and distinct value counts
- Naming conventions and type information

## Prerequisites

- Completed [01_basic_connection.ipynb](01_basic_connection.ipynb) or familiar with database connections
- Understanding of basic data types (integers, strings, dates)
- Familiarity with column analysis concepts

## What You'll Learn

- How to analyze individual columns for their likely purpose
- How to interpret confidence scores and reasoning
- How to understand type-specific statistics (numeric, string, datetime)
- How to detect enumeration columns and flags
- How to identify IDs, amounts, quantities, and percentages

## Environment Verification

In [ ]:
import sys
from pathlib import Path

print(f"Python version: {sys.version}")

# Find repository root and add to path
def find_repo_root():
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    if current.name == "notebooks":
        return current.parent.parent
    return current

repo_root = find_repo_root()
sys.path.insert(0, str(repo_root))
print(f"Repository root: {repo_root}")

from examples.shared.notebook_helpers import verify_notebook_environment
verify_notebook_environment()

## Section 1: Setup and Connection

First, let's connect to our test database and set up the Column Analyzer.

In [ ]:
from sqlalchemy import create_engine
from src.inference.columns import ColumnAnalyzer
from examples.shared.notebook_helpers import print_table

# Connect to test database
test_db_path = repo_root / "examples" / "test_database" / "example.db"
engine = create_engine(f"sqlite:///{test_db_path}")

# Create analyzer
analyzer = ColumnAnalyzer(engine)

print("✓ Column Analyzer ready")
print(f"✓ Connected to: {test_db_path.name}")

## Section 2: Analyzing an ID Column

Let's start by analyzing a primary key column. ID columns typically have:
- Unique values (distinct count = row count)
- Integer data types
- Sequential or auto-incrementing values
- Names ending in `_id`, `id`, `_key`, etc.

In [ ]:
# Analyze the customer_id column
analysis = analyzer.analyze_column(
    column_name="customer_id",
    table_name="customers",
    schema_name="main"  # SQLite uses 'main' as default schema
)

print(f"Column: {analysis.column_name}")
print(f"Table: {analysis.table_name}")
print(f"Data Type: {analysis.data_type}")
print()
print("=== Inference Results ===")
print(f"Inferred Purpose: {analysis.inferred_purpose.value.upper()}")
print(f"Confidence: {analysis.confidence:.0%}")
print(f"Reasoning: {analysis.reasoning}")
print()
print("=== Statistics ===")
print(f"Total Rows: {analysis.total_rows}")
print(f"Distinct Values: {analysis.distinct_count}")
print(f"Null Count: {analysis.null_count}")
print(f"Is Enumeration: {analysis.is_enum}")

**Understanding the results**:

The analyzer correctly identifies `customer_id` as an **ID** column because:
- The name ends with `_id` (strong naming pattern)
- Integer data type is typical for IDs
- All values are unique (distinct count = total rows)

**Key insight**: High confidence (85%+) indicates strong pattern matching. Lower confidence means the inference is more speculative.

## Section 3: Analyzing an Enumeration Column

Enum columns have a small set of repeated values. They often represent:
- Status codes (Active, Inactive, Pending)
- Category types (Electronics, Clothing, Food)
- Fixed options (Yes/No, True/False)

In [ ]:
# Analyze the status column from orders table
analysis = analyzer.analyze_column(
    column_name="status",
    table_name="orders",
    schema_name="main"
)

print(f"Column: {analysis.column_name}")
print(f"Inferred Purpose: {analysis.inferred_purpose.value.upper()}")
print(f"Confidence: {analysis.confidence:.0%}")
print(f"Reasoning: {analysis.reasoning}")
print(f"\nIs Enumeration: {analysis.is_enum}")
print(f"Distinct Values: {analysis.distinct_count}")

# Show string statistics if available
if analysis.string_stats:
    print("\n=== Top Values ===")
    for value, freq in analysis.string_stats.top_values[:5]:
        print(f"  '{value}': {freq} occurrences")

**Understanding the results**:

The analyzer identifies columns as **STATUS** or **ENUM** based on:
- Low cardinality (few distinct values)
- Name patterns containing 'status', 'state', 'type'
- Less than 10% of rows have unique values

**Key insight**: The `is_enum` flag helps distinguish between:
- True enumerations (status, type) with very few values
- High-cardinality columns that shouldn't be treated as enums

## Section 4: Analyzing Numeric Columns

Numeric columns can represent many things:
- **Amounts**: Prices, totals, balances
- **Quantities**: Counts, units, items
- **Percentages**: Rates, ratios, proportions
- **IDs**: Foreign keys, sequence numbers

The analyzer uses statistics (min, max, mean) and naming patterns to distinguish between these.

In [ ]:
# Analyze a numeric column - total_amount from orders
analysis = analyzer.analyze_column(
    column_name="total_amount",
    table_name="orders",
    schema_name="main"
)

print(f"Column: {analysis.column_name}")
print(f"Inferred Purpose: {analysis.inferred_purpose.value.upper()}")
print(f"Confidence: {analysis.confidence:.0%}")
print(f"Reasoning: {analysis.reasoning}")

# Show numeric statistics
if analysis.numeric_stats:
    stats = analysis.numeric_stats
    print("\n=== Numeric Statistics ===")
    print(f"  Min: {stats.min_value}")
    print(f"  Max: {stats.max_value}")
    print(f"  Mean: {stats.mean_value:.2f}" if stats.mean_value else "  Mean: N/A")
    print(f"  Is Integer: {stats.is_integer}")

In [ ]:
# Compare with a quantity column
analysis = analyzer.analyze_column(
    column_name="quantity",
    table_name="order_items",
    schema_name="main"
)

print(f"Column: {analysis.column_name}")
print(f"Inferred Purpose: {analysis.inferred_purpose.value.upper()}")
print(f"Confidence: {analysis.confidence:.0%}")
print(f"Reasoning: {analysis.reasoning}")

if analysis.numeric_stats:
    stats = analysis.numeric_stats
    print(f"\nRange: {stats.min_value} to {stats.max_value}")
    print(f"Is Integer: {stats.is_integer}")

**Understanding the results**:

The analyzer distinguishes between numeric purposes:
- **AMOUNT**: Names containing 'amt', 'amount', 'price', 'total', 'cost'
- **QUANTITY**: Names containing 'qty', 'quantity', 'count', 'units'
- **PERCENTAGE**: Values in 0-100 or 0-1 range, names with 'pct', 'rate'

**Key insight**: Numeric statistics help validate inferences. A column named 'quantity' with decimal values might indicate a different purpose than expected.

## Section 5: Analyzing DateTime Columns

DateTime columns often represent:
- Creation timestamps (`created_at`, `created_date`)
- Modification timestamps (`updated_at`, `modified_date`)
- Event times (`order_date`, `shipped_date`)

The analyzer examines date ranges and time components.

**Note**: The `has_time_component` statistic detects whether datetime values include non-midnight times. This feature is fully supported on SQL Server but returns `False` for SQLite databases due to SQLite's variable datetime storage formats.

In [ ]:
# Analyze order_date column
analysis = analyzer.analyze_column(
    column_name="order_date",
    table_name="orders",
    schema_name="main"
)

print(f"Column: {analysis.column_name}")
print(f"Inferred Purpose: {analysis.inferred_purpose.value.upper()}")
print(f"Confidence: {analysis.confidence:.0%}")
print(f"Reasoning: {analysis.reasoning}")

# Show datetime statistics
if analysis.datetime_stats:
    stats = analysis.datetime_stats
    print("\n=== DateTime Statistics ===")
    print(f"  Earliest: {stats.min_date}")
    print(f"  Latest: {stats.max_date}")
    print(f"  Range: {stats.date_range_days} days" if stats.date_range_days else "  Range: N/A")
    print(f"  Has Time Component: {stats.has_time_component}")

## Section 6: Analyzing Multiple Columns

Let's analyze all columns in a table to get a complete picture of its structure and purposes.

In [ ]:
from sqlalchemy import inspect

# Get all columns from customers table
inspector = inspect(engine)
columns = inspector.get_columns("customers")

print("=== Column Analysis: customers table ===")
print()

results = []
for col in columns:
    analysis = analyzer.analyze_column(
        column_name=col['name'],
        table_name="customers",
        schema_name="main"
    )
    results.append([
        analysis.column_name,
        analysis.data_type,
        analysis.inferred_purpose.value,
        f"{analysis.confidence:.0%}",
        "Yes" if analysis.is_enum else "No"
    ])

print_table(results, ["Column", "Type", "Purpose", "Confidence", "Enum?"])

## Section 7: Understanding Confidence Scores

Confidence scores help you understand how reliable the inference is:

| Score | Meaning | Action |
|-------|---------|--------|
| 90%+ | Very high confidence | Trust the inference |
| 75-89% | High confidence | Likely correct, verify if critical |
| 60-74% | Moderate confidence | Consider manually verifying |
| <60% | Low confidence | Manual review recommended |

**Factors that increase confidence**:
- Strong naming patterns (`_id`, `_date`, `_amt`)
- Matching data characteristics (unique IDs, limited enum values)
- Value range validation (percentages in 0-100)

In [ ]:
# Example: Analyze a column with lower confidence
analysis = analyzer.analyze_column(
    column_name="name",
    table_name="customers",
    schema_name="main"
)

print(f"Column: {analysis.column_name}")
print(f"Inferred Purpose: {analysis.inferred_purpose.value}")
print(f"Confidence: {analysis.confidence:.0%}")
print(f"Reasoning: {analysis.reasoning}")
print()
print("Note: Generic column names like 'name' may have lower confidence")
print("because they don't match specific purpose patterns.")

## Summary

**What we covered**:
- ✓ Analyzed ID columns and understood uniqueness patterns
- ✓ Detected enumeration and status columns
- ✓ Distinguished between amounts, quantities, and percentages
- ✓ Examined datetime columns and their statistics
- ✓ Interpreted confidence scores and reasoning

**Key concepts**:
- **Inferred Purpose**: The likely meaning of a column (id, enum, amount, etc.)
- **Confidence Score**: How reliable the inference is (0-100%)
- **Reasoning**: Explanation of why a purpose was inferred
- **Type-Specific Statistics**: Numeric, string, or datetime analysis

**Best practices**:
- ✓ Use column analysis to understand legacy databases with cryptic names
- ✓ Review low-confidence inferences manually
- ✓ Combine with sample data inspection for validation
- ⚠️ Don't blindly trust inferences - they're educated guesses

## Next Steps

**Continue learning**:
- 📓 [05_documentation_cache.ipynb](05_documentation_cache.ipynb) - Cache and export database documentation
- 📓 [03_advanced_patterns.ipynb](03_advanced_patterns.ipynb) - Relationship inference and error handling

**Explore further**:
- 💡 Try analyzing columns in your own databases
- 💡 Combine column analysis with sample data for deeper understanding
- 💡 Use inferred purposes to generate better documentation

**Need help?**
- 🐛 [Report issues](https://github.com/anthropics/dbmcp/issues)
- 💬 [Join discussions](https://github.com/anthropics/dbmcp/discussions)